# 02 — Longitudinal Synthesis (Part I)

Build Part I draft from structured extraction JSON only. Every factual sentence must cite `[src. Pg. N]`.

In [ ]:
import json
from pathlib import Path

import sys
from pathlib import Path

_root = Path.cwd().resolve()
if _root.name == "notebooks":
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.nb_utils import setup_project_root
from src.gemini_json import generate_json
from src.load_pages import DEFAULT_OUTPUT_DIR, load_corpus
from src.prompts import SYNTHESIS_SYSTEM
from src.schemas import SynthesisResult

ROOT = setup_project_root()
corpus = load_corpus()


def load_extractions():
    out = {}
    for name in ("ortho", "neuro", "billing", "baseline"):
        path = DEFAULT_OUTPUT_DIR / f"{name}.json"
        if not path.exists():
            raise FileNotFoundError(f"Run 01_extraction_agents first. Missing {path}")
        out[name] = json.loads(path.read_text(encoding="utf-8"))
    return out


extractions = load_extractions()
extractions

In [ ]:
user_prompt = (
    f"case_id: {corpus.case_id}\n"
    f"date_of_loss: {corpus.date_of_loss}\n\n"
    f"Structured extractions (use only this data):\n"
    f"{json.dumps(extractions, indent=2)}"
)

synthesis, usage = generate_json(SYNTHESIS_SYSTEM, user_prompt, SynthesisResult)
print("Token usage:", usage)
synthesis

In [ ]:
lines = ["# Part I — Clinical & Financial Summary", ""]
for section in synthesis.sections:
    lines.append(f"## {section.title}")
    lines.append("")
    lines.append(section.body_markdown)
    lines.append("")

draft = "\n".join(lines)
draft_path = DEFAULT_OUTPUT_DIR / "part1_draft.md"
draft_path.write_text(draft, encoding="utf-8")
print(f"Saved {draft_path}")
print(draft[:2000])